In [ ]:
pip install librosa soundfile numpy matplotlib tensorflow


In [ ]:
import os
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dropout, Dense, Bidirectional, BatchNormalization, Activation
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

data_path = "path/to/data"
model_filename = 'emotion.h5'


In [ ]:
import os
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

emotions = {'01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
            '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'}

x_data, y_labels = [], []

data_path = "path/to/data"

if not os.path.exists(data_path):
    raise ValueError(f"Error: The path {data_path} does not exist.")
else:
    print("Contents of the data_path:", os.listdir(data_path))

def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=None)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        mfcc = np.mean(mfcc.T, axis=0)
        return mfcc
    except Exception as e:
        print(f"Error extracting features from {file_path}: {e}")
        return np.array([])

for actor_folder in os.listdir(data_path):
    actor_path = os.path.join(data_path, actor_folder)
    if os.path.isdir(actor_path):
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):
                try:
                    emotion_code = file.split("-")[2]
                    emotion_label = emotions.get(emotion_code, 'unknown')

                    feature = extract_features(os.path.join(actor_path, file))

                    if feature.shape[0] > 0:
                        x_data.append(feature)
                        y_labels.append(emotion_label)

                    print(f"Processed {file} in {actor_folder}: Feature shape {feature.shape}")

                except Exception as e:
                    print(f"Error processing {file} in {actor_folder}: {e}")

x_data = np.array(x_data)
y_labels = np.array(y_labels)

label_encoder = LabelEncoder()
y_labels = label_encoder.fit_transform(y_labels)

if len(x_data) > 0 and len(y_labels) > 0:
    x_train, x_test, y_train, y_test = train_test_split(x_data, y_labels, test_size=0.2, random_state=42)
    print(f"Training data shape: {x_train.shape}")
    print(f"Test data shape: {x_test.shape}")
else:
    print("Error: No data available to split. Check data path and feature extraction.")


In [ ]:
x_train = np.expand_dims(x_train, axis=2)
x_test = np.expand_dims(x_test, axis=2)

print(f"Reshaped training data shape: {x_train.shape}")
print(f"Reshaped testing data shape: {x_test.shape}")


In [ ]:
model = Sequential()
model.add(LSTM(128, input_shape=(x_train.shape[1], 1), return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(64))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(len(np.unique(y_labels)), activation='softmax'))

model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
history = model.fit(x_train, y_train, epochs=200, batch_size=32, validation_data=(x_test, y_test))


In [ ]:
model.save('emotion_model.h5')
print("Model saved to emotion_model.h5")


In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=1)

print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


In [ ]:
from IPython.display import Audio
audio_file = "path/to/sample_audio.wav"
Audio(audio_file)


In [ ]:
def predict_emotion(model, audio_file):
    feature = extract_features(audio_file)
    if feature.shape[0] > 0:
        feature = np.expand_dims(feature, axis=0)
        feature = np.expand_dims(feature, axis=2)

        prediction = model.predict(feature)
        emotion_index = np.argmax(prediction)
        emotion = label_encoder.inverse_transform([emotion_index])

        return emotion[0]
    else:
        return "Error: Unable to extract features"

predicted_emotion = predict_emotion(model, audio_file)
print(f"Predicted Emotion: {predicted_emotion}")
